# EcoShield AI - La Smart City sous Haute Protection 🛡️

Ce notebook contient l'intégralité du code backend d'EcoShield AI, permettant :
1. **Génération de données synthétiques** pour l'énergie et l'eau.
2. **Simulation d'attaques** cyber (FDI - False Data Injection).
3. **Entraînement de modèles PyTorch** (LSTM pour l'optimisation, Autoencodeur pour la détection).
4. **Simulation temps réel** du pipeline de mitigation des attaques.

---

## 1. Importation des librairies

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
%matplotlib inline

## 2. Génération de Données Synthétiques (Smart City API)
Génération d'un historique normal de consommation d'une ville (Électricité & Eau) avec cycles et bruits habituels.

In [ ]:
def generate_synthetic_data(days=30):
    timestamps = pd.date_range(start="2023-01-01", periods=days*24, freq="H")
    
    base_energy = 500  # kW
    base_water = 100   # m3
    
    hourly_pattern = np.array([
        0.6, 0.5, 0.5, 0.5, 0.6, 0.8, 1.2, 1.5, 1.8, 1.6, 1.4, 1.3,
        1.3, 1.3, 1.4, 1.5, 1.7, 1.9, 2.0, 1.9, 1.6, 1.3, 0.9, 0.7
    ])
    
    repeated_pattern = np.tile(hourly_pattern, days)
    
    energy_noise = np.random.normal(0, 0.1, len(timestamps))
    water_noise = np.random.normal(0, 0.1, len(timestamps))
    
    seasonal_variation = np.sin(np.linspace(0, 2 * np.pi, len(timestamps))) * 0.2 + 1.0
    
    energy_consumption = base_energy * repeated_pattern * seasonal_variation * (1 + energy_noise)
    water_consumption = base_water * repeated_pattern * seasonal_variation * (1 + water_noise)
    
    energy_consumption = np.maximum(energy_consumption, 0)
    water_consumption = np.maximum(water_consumption, 0)
    
    df = pd.DataFrame({
        "timestamp": timestamps,
        "energy_consumption_kw": energy_consumption,
        "water_consumption_m3": water_consumption
    })
    
    df.set_index("timestamp", inplace=True)
    return df

clean_df = generate_synthetic_data(days=60)
print(f"Données synthétiques générées: {len(clean_df)} heures.")
clean_df[['energy_consumption_kw', 'water_consumption_m3']][:72].plot(figsize=(12,4), title="3 Jours de consommation normale")

## 3. Préparation des Datasets pour PyTorch

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_length):
        self.data = data
        self.seq_length = seq_length
    def __len__(self):
        return len(self.data) - self.seq_length
    def __getitem__(self, index):
        x = self.data[index:index+self.seq_length]
        y = self.data[index+self.seq_length]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

class AutoencoderDataset(Dataset):
    def __init__(self, data, seq_length):
        self.data = data
        self.seq_length = seq_length
    def __len__(self):
        return len(self.data) - self.seq_length + 1
    def __getitem__(self, index):
        x = self.data[index:index+self.seq_length]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(x, dtype=torch.float32)

sequence_length = 24
data_values = clean_df.values
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data_values)

train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
val_data = scaled_data[train_size:]

train_lstm_loader = DataLoader(TimeSeriesDataset(train_data, sequence_length), batch_size=32, shuffle=True)
train_ae_loader = DataLoader(AutoencoderDataset(train_data, sequence_length), batch_size=32, shuffle=True)
print("Dataloaders prêts.")

## 4. Architectures des Modèles (LSTM & Autoencodeur)

In [ ]:
class LSTMForescaster(nn.Module):
    """ The Guardian : Prédit la valeur suivante pour remplacer les anomalies """
    def __init__(self, input_size=2, hidden_size=64, num_layers=2, output_size=2):
        super(LSTMForescaster, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

class TimeSeriesAutoencoder(nn.Module):
    """ The Watchdog : Reconstruit la séquence, une grosse erreur = Attaque FDI """
    def __init__(self, input_size=2, sequence_length=24, hidden_size=32):
        super(TimeSeriesAutoencoder, self).__init__()
        self.sequence_length = sequence_length
        self.encoder = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, input_size, batch_first=True)
        
    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        hidden = hidden.squeeze(0).unsqueeze(1).repeat(1, self.sequence_length, 1)
        decoded, _ = self.decoder(hidden)
        return decoded

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lstm_model = LSTMForescaster().to(device)
ae_model = TimeSeriesAutoencoder().to(device)
print(f"Modèles instanciés sur {device}")

## 5. Entraînement Rapide

In [ ]:
def train(model, dataloader, epochs=5):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.5f}")
    return model

print("--- Entraînement LSTM (Forecaster) ---")
train(lstm_model, train_lstm_loader, epochs=5)

print("\n--- Entraînement Autoencodeur (Détecteur) ---")
train(ae_model, train_ae_loader, epochs=5)

## 6. Simulation d'Attaque (FDI) & Détection
Injection d'un Spike (Pic énorme) et réaction du modèle.

In [ ]:
# Générer de nouvelles données saines
test_df = generate_synthetic_data(days=3)
attack_df = test_df.copy()

# Injecter un gros Spike à l'index 48
attack_idx = 48
attack_df.iloc[attack_idx, 0] = attack_df.iloc[attack_idx, 0] * 3.0 # Fausse donnée x3

# Préparation série temporelle attaqué (jour 1 et 2)
seq_attacked = attack_df.values[attack_idx-23:attack_idx+1] # 24h 
seq_scaled = scaler.transform(seq_attacked)

# Détection avec l'Autoencodeur
ae_model.eval()
with torch.no_grad():
    tensor_in = torch.tensor(seq_scaled, dtype=torch.float32).unsqueeze(0).to(device)
    reconstructed = ae_model(tensor_in)
    # Erreur MSE du dernier point
    mse_error_point = torch.nn.functional.mse_loss(reconstructed[0, -1], tensor_in[0, -1]).item()

print(f"==> Erreur de reconstruction sur le point piraté : {mse_error_point:.4f}")
if mse_error_point > 0.05: # Seuil d'anomalie basique
    print("🚨 ALERTE: Attaque FDI Détectée ! Rejet de la donnée du capteur.")
    
    # Mitigation par le LSTM (On lui donne les 24h précédentes saines pour prédire la vraie valeur)
    lstm_model.eval()
    seq_saine_scaled = scaler.transform(attack_df.values[attack_idx-24:attack_idx])
    with torch.no_grad():
        tensor_hist = torch.tensor(seq_saine_scaled, dtype=torch.float32).unsqueeze(0).to(device)
        prediction_scaled = lstm_model(tensor_hist)
        
    valeur_predite = scaler.inverse_transform(prediction_scaled.cpu().numpy())[0]
    
    print(f"❌ Fausse Donnée reçue : {attack_df.iloc[attack_idx, 0]:.1f} kW")
    print(f"✅ Prédiction Sécurisée IA (Guardia) : {valeur_predite[0]:.1f} kW")
    print(f"🔋 Consommation Optimisée (-30%) : {valeur_predite[0] * 0.7:.1f} kW")